In [1]:
from pathlib import Path
import polars as pl

In [2]:
PROJECT_ROOT = Path.cwd().parent

CLEAN_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "trips_clean.parquet"
)

print("Project root:", PROJECT_ROOT)
print("Cleaned dataset:", CLEAN_FILE)
print("File exists:", CLEAN_FILE.exists())

Project root: E:\SLIIT_Bacholer\Datathon2026\UrbanFlow-Datathon
Cleaned dataset: E:\SLIIT_Bacholer\Datathon2026\UrbanFlow-Datathon\data\processed\trips_clean.parquet
File exists: True


In [3]:
df = pl.scan_parquet(CLEAN_FILE)

schema = df.collect_schema()

print("Number of columns:", len(schema))
print("\nColumns and data types:")

for name, dtype in schema.items():
    print(f"{name}: {dtype}")

Number of columns: 22

Columns and data types:
provider_code: Int64
pickup_timestamp: Datetime(time_unit='us', time_zone=None)
dropoff_timestamp: Datetime(time_unit='us', time_zone=None)
rider_count: Int64
distance_miles: Float64
rate_class_id: Int64
offline_record_flag: String
origin_loc_id: Int64
dest_loc_id: Int64
fare_settlement_method: Int64
base_fare: Float64
surcharge_misc: Float64
transit_tax: Float64
driver_tip_payment: Float64
toll_total: Float64
service_improvement_fee: Float64
charge_total: Float64
zone_congestion_fee: Float64
Airport_fee: Float64
congestion_relief_fee: Float64
trip_duration_minutes: Float64
speed_mph: Float64


In [ ]:
# Splitting data

In [6]:
monthly_counts = (
    df
    .with_columns(
        pl.col("pickup_timestamp")
        .dt.strftime("%Y-%m")
        .alias("month")
    )
    .group_by("month")
    .agg(
        pl.len().alias("rows")
    )
    .sort("month")
    .collect()
)

print(monthly_counts)
split_preview = (
    df
    .with_columns(
        pl.when(
            pl.col("pickup_timestamp") < pl.datetime(2026, 1, 1)
        )
        .then(pl.lit("Train"))
        .when(
            pl.col("pickup_timestamp") < pl.datetime(2026, 3, 1)
        )
        .then(pl.lit("Validation"))
        .otherwise(pl.lit("Test"))
        .alias("split")
    )
    .group_by("split")
    .agg(
        pl.len().alias("rows"),
        pl.col("pickup_timestamp").min().alias("start_date"),
        pl.col("pickup_timestamp").max().alias("end_date"),
    )
    .sort("start_date")
    .collect()
)

print(split_preview)

shape: (12, 2)
┌─────────┬─────────┐
│ month   ┆ rows    │
│ ---     ┆ ---     │
│ str     ┆ u32     │
╞═════════╪═════════╡
│ 2025-04 ┆ 3672531 │
│ 2025-05 ┆ 4093408 │
│ 2025-06 ┆ 3870028 │
│ 2025-07 ┆ 3495214 │
│ 2025-08 ┆ 3181272 │
│ …       ┆ …       │
│ 2025-11 ┆ 3644999 │
│ 2025-12 ┆ 4051172 │
│ 2026-01 ┆ 3518171 │
│ 2026-02 ┆ 3211948 │
│ 2026-03 ┆ 3764245 │
└─────────┴─────────┘
shape: (3, 4)
┌────────────┬──────────┬─────────────────────┬─────────────────────┐
│ split      ┆ rows     ┆ start_date          ┆ end_date            │
│ ---        ┆ ---      ┆ ---                 ┆ ---                 │
│ str        ┆ u32      ┆ datetime[μs]        ┆ datetime[μs]        │
╞════════════╪══════════╪═════════════════════╪═════════════════════╡
│ Train      ┆ 33792312 ┆ 2025-04-01 00:00:00 ┆ 2025-12-31 23:59:57 │
│ Validation ┆ 6730119  ┆ 2026-01-01 00:00:00 ┆ 2026-02-28 23:59:59 │
│ Test       ┆ 3764245  ┆ 2026-03-01 00:00:00 ┆ 2026-03-31 23:59:59 │
└────────────┴──────────┴────────────

In [7]:
original_columns = [
    "provider_code",
    "pickup_timestamp",
    "dropoff_timestamp",
    "rider_count",
    "distance_miles",
    "rate_class_id",
    "offline_record_flag",
    "origin_loc_id",
    "dest_loc_id",
    "fare_settlement_method",
    "base_fare",
    "surcharge_misc",
    "transit_tax",
    "driver_tip_payment",
    "toll_total",
    "service_improvement_fee",
    "charge_total",
    "zone_congestion_fee",
    "Airport_fee",
    "congestion_relief_fee",
]

print("Columns to save:", len(original_columns))

for column in original_columns:
    print(column)

Columns to save: 20
provider_code
pickup_timestamp
dropoff_timestamp
rider_count
distance_miles
rate_class_id
offline_record_flag
origin_loc_id
dest_loc_id
fare_settlement_method
base_fare
surcharge_misc
transit_tax
driver_tip_payment
toll_total
service_improvement_fee
charge_total
zone_congestion_fee
Airport_fee
congestion_relief_fee


In [8]:
train_df = (
    df
    .filter(
        pl.col("pickup_timestamp") < pl.datetime(2026, 1, 1)
    )
    .select(original_columns)
)

validation_df = (
    df
    .filter(
        (pl.col("pickup_timestamp") >= pl.datetime(2026, 1, 1))
        &
        (pl.col("pickup_timestamp") < pl.datetime(2026, 3, 1))
    )
    .select(original_columns)
)

test_df = (
    df
    .filter(
        pl.col("pickup_timestamp") >= pl.datetime(2026, 3, 1)
    )
    .select(original_columns)
)

print("Train, validation, and test LazyFrames created.")
print("Nothing has been saved yet.")

Train, validation, and test LazyFrames created.
Nothing has been saved yet.


In [9]:
split_check = pl.DataFrame(
    {
        "split": ["Train", "Validation", "Test"],
        "rows": [
            train_df.select(pl.len()).collect().item(),
            validation_df.select(pl.len()).collect().item(),
            test_df.select(pl.len()).collect().item(),
        ],
        "start_date": [
            train_df.select(pl.col("pickup_timestamp").min()).collect().item(),
            validation_df.select(pl.col("pickup_timestamp").min()).collect().item(),
            test_df.select(pl.col("pickup_timestamp").min()).collect().item(),
        ],
        "end_date": [
            train_df.select(pl.col("pickup_timestamp").max()).collect().item(),
            validation_df.select(pl.col("pickup_timestamp").max()).collect().item(),
            test_df.select(pl.col("pickup_timestamp").max()).collect().item(),
        ],
    }
)

print(split_check)

shape: (3, 4)
┌────────────┬──────────┬─────────────────────┬─────────────────────┐
│ split      ┆ rows     ┆ start_date          ┆ end_date            │
│ ---        ┆ ---      ┆ ---                 ┆ ---                 │
│ str        ┆ i64      ┆ datetime[μs]        ┆ datetime[μs]        │
╞════════════╪══════════╪═════════════════════╪═════════════════════╡
│ Train      ┆ 33792312 ┆ 2025-04-01 00:00:00 ┆ 2025-12-31 23:59:57 │
│ Validation ┆ 6730119  ┆ 2026-01-01 00:00:00 ┆ 2026-02-28 23:59:59 │
│ Test       ┆ 3764245  ┆ 2026-03-01 00:00:00 ┆ 2026-03-31 23:59:59 │
└────────────┴──────────┴─────────────────────┴─────────────────────┘


In [10]:
SPLIT_DIR = PROJECT_ROOT / "data" / "splits"

TRAIN_FILE = SPLIT_DIR / "train" / "trips_train.parquet"
VALIDATION_FILE = SPLIT_DIR / "validation" / "trips_validation.parquet"
TEST_FILE = SPLIT_DIR / "test" / "trips_test.parquet"

TRAIN_FILE.parent.mkdir(parents=True, exist_ok=True)
VALIDATION_FILE.parent.mkdir(parents=True, exist_ok=True)
TEST_FILE.parent.mkdir(parents=True, exist_ok=True)

print("Saving train split...")
train_df.sink_parquet(str(TRAIN_FILE), compression="zstd")

print("Saving validation split...")
validation_df.sink_parquet(str(VALIDATION_FILE), compression="zstd")

print("Saving test split...")
test_df.sink_parquet(str(TEST_FILE), compression="zstd")

print("\nAll split datasets saved successfully.")
print("Train:", TRAIN_FILE)
print("Validation:", VALIDATION_FILE)
print("Test:", TEST_FILE)

Saving train split...
Saving validation split...
Saving test split...

All split datasets saved successfully.
Train: E:\SLIIT_Bacholer\Datathon2026\UrbanFlow-Datathon\data\splits\train\trips_train.parquet
Validation: E:\SLIIT_Bacholer\Datathon2026\UrbanFlow-Datathon\data\splits\validation\trips_validation.parquet
Test: E:\SLIIT_Bacholer\Datathon2026\UrbanFlow-Datathon\data\splits\test\trips_test.parquet


In [11]:
saved_files = {
    "Train": TRAIN_FILE,
    "Validation": VALIDATION_FILE,
    "Test": TEST_FILE,
}

expected_rows = {
    "Train": 33_792_312,
    "Validation": 6_730_119,
    "Test": 3_764_245,
}

print("FINAL SAVED SPLIT VERIFICATION\n")

all_passed = True

for name, file_path in saved_files.items():

    if not file_path.exists():
        print(f"{name}: FAIL - file not found")
        all_passed = False
        continue

    saved_df = pl.scan_parquet(file_path)

    row_count = saved_df.select(
        pl.len()
    ).collect().item()

    column_count = len(saved_df.collect_schema())

    min_date = saved_df.select(
        pl.col("pickup_timestamp").min()
    ).collect().item()

    max_date = saved_df.select(
        pl.col("pickup_timestamp").max()
    ).collect().item()

    rows_ok = row_count == expected_rows[name]
    columns_ok = column_count == 20

    status = "PASS" if rows_ok and columns_ok else "FAIL"

    if status == "FAIL":
        all_passed = False

    print(f"{name}: {status}")
    print(f"  Rows: {row_count:,}")
    print(f"  Columns: {column_count}")
    print(f"  Start: {min_date}")
    print(f"  End: {max_date}")
    print()

total_rows = sum(expected_rows.values())

print(f"Expected total rows: {total_rows:,}")

if all_passed:
    print("\nFINAL RESULT: SAVED SPLITS VERIFIED")
else:
    print("\nFINAL RESULT: SPLIT VERIFICATION FAILED")

FINAL SAVED SPLIT VERIFICATION

Train: PASS
  Rows: 33,792,312
  Columns: 20
  Start: 2025-04-01 00:00:00
  End: 2025-12-31 23:59:57

Validation: PASS
  Rows: 6,730,119
  Columns: 20
  Start: 2026-01-01 00:00:00
  End: 2026-02-28 23:59:59

Test: PASS
  Rows: 3,764,245
  Columns: 20
  Start: 2026-03-01 00:00:00
  End: 2026-03-31 23:59:59

Expected total rows: 44,286,676

FINAL RESULT: SAVED SPLITS VERIFIED
